# M4 · Visión como *input* — launcher

Este notebook **solo orquesta** el módulo `M4/vision/`. No reimplementa la visión, el harness ni la app.

- **Qué hace:** lee las imágenes del eval set de M3 desde Google Drive (`M4/data/imagenes/`) y genera **2 JSON** de transcripción: uno para el caso normal (`limpio`) y otro para el caso realista/reto (`foto_realista`).
- **No consume API de Groq:** el VLM (`Qwen/Qwen2-VL-2B-Instruct`) corre localmente en la GPU de Colab.


## 1 · Clonar repositorio e instalar dependencias

In [ ]:
RAMA = "LuisM4"

!git clone --branch {RAMA} --single-branch https://github.com/luisNP21/Topicos-IA.git /content/repo
%cd /content/repo
!git pull
!pip install -r M4/vision/requirements.txt -q

## 2 · Montar Drive y definir `PROJECT_ROOT`

Lee las imágenes de `<PROJECT_ROOT>/M4/data/imagenes/` y guarda los 2 JSON en `<PROJECT_ROOT>/M4/vision/predictions/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud")
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

with open("/content/repo/.env", "w") as f:
    f.write(f"PROJECT_ROOT={PROJECT_ROOT}\n")

print("PROJECT_ROOT:", PROJECT_ROOT)
for nivel in ("limpio", "foto_realista"):
    csv_path = PROJECT_ROOT / f"M4/data/imagenes/casos/{nivel}.csv"
    print(f"  CSV {nivel}:", csv_path.exists(), "->", csv_path)

## 3 · Generar los 2 JSON: Caso Normal (`limpio`) vs. Caso Realista/Reto (`foto_realista`)

- Si `LIMITE = 1`, solo transcribe el primer ejemplo (`caso_01_p1`) en cada JSON (prueba ultrarrápida).
- Si `LIMITE = None`, transcribe los 15 casos (`caso_01` a `caso_15`) del eval set de M3.
- Si quieres que también genere el cache NER (`M2/predictions/vision_vlm_*.json`), pon `CORRER_NER = True`.

In [ ]:
LIMITE = 1          # Pon None para correr los 15 casos de M3, o 1 para probar solo con caso_01
CORRER_NER = False  # Pon True si tienes el adaptador de M1 en Drive y quieres el cache NER

flag_limite = f"--limite {LIMITE}" if LIMITE else ""
flag_ner = "--ner" if CORRER_NER else ""

# 1) JSON Normal (limpio)
!python M4/vision/run_vision.py --config M4/vision/config.yaml \
    --backend vlm \
    --manifest M4/data/imagenes/casos/limpio.csv \
    --nivel limpio {flag_limite} {flag_ner}

# 2) JSON Realista / Reto (foto_realista)
!python M4/vision/run_vision.py --config M4/vision/config.yaml \
    --backend vlm \
    --manifest M4/data/imagenes/casos/foto_realista.csv \
    --nivel foto_realista {flag_limite} {flag_ner}

## 4 · Verificar los 2 JSON generados en Drive

In [ ]:
for nombre in ("transcripciones_vlm_limpio.json", "transcripciones_vlm_foto_realista.json"):
    ruta = PROJECT_ROOT / "M4/vision/predictions" / nombre
    print("=" * 70)
    print(nombre, "| existe:", ruta.exists())
    if ruta.exists():
        datos = json.loads(ruta.read_text(encoding="utf-8"))
        print("Casos incluidos:", list(datos.keys()))
        primero = next(iter(datos.values()))
        print("Ejemplo:", json.dumps(primero, ensure_ascii=False, indent=2)[:800])